<!-- nav -->
[← 8.12 · POSIX ERE](08_12_POSIX_ERE.ipynb) · [Home](../../README.md) · [8.14 · SystemVerilog Patterns →](08_14_SystemVerilog_Patterns.ipynb)

# 8.13 · SystemVerilog Types

Tutorial 8 met SystemVerilog's sublanguages once each: a cast's type, `int'(x)`, was a term of `SystemVerilog Types`
(08.1). This page takes that dialect on its own, from the data types a reader of IEEE 1800 knows (clauses 6 and 7) to
their models: a keyword type, a vector with packed dimensions, a structure, a union and an enumeration, each written in
the builders' fluent calls beside the text SystemVerilog writes, which the family's printer gives back. Then what is
done with them: validated, with one refused; read by the expressions that hold them, a cast, `$bits` and an assignment
pattern, whose rules convert a value to some of them; given no domain and no value of their own; and the editions
of Verilog and SystemVerilog that have each kind, read as data.

Like the rest of Part I, it starts from `dialects()`, a store with every dialect installed and defined. `name` and
`literal` write a name and a literal of SystemVerilog's `Expressions` (tutorials 3 and 4), wherever a type holds one: a
dimension's bounds are expressions. `refused` gives what a simplifier says of an expression, its value where its residual is a literal, or why it has none, as tutorial 8's does; `attributes` lists a form's attributes, `name=value`, a helper by its kind (tutorial
6, 06.5).

In [1]:
from mbse.Expressions.SystemVerilog import Expressions as SystemVerilog, Types as SVTypes
from mbse.Expressions.Framework import Domains, Standards, Symbolics, Terms
from toolkit import dialects

store = dialects()
S = store.Expressions
Types = SVTypes.Expressions.DIALECT  # the dialect, as the framework reads it
name = lambda spelling: lambda d: d.nameexpression(lambda n: n.name(lambda a: a.identifier(lambda i: i.spelling(spelling))))
literal = lambda value: lambda d: d.literal(lambda l: l.value(lambda v: v.as_scalar(value)))


def refused(simplify, expression):
    """What `simplify` says of `expression`: its value, a simplification's where its residual is a literal, or why it has
    none (`NotEvaluated`, a `ValueError`)."""
    try:
        result = simplify(expression)
        return str(result.evaluate() if hasattr(result, "evaluate") else result)
    except ValueError as error:
        return str(error)


def attributes(form):
    """A form's attributes, each `name=value`: a helper as its kind, helpers in brackets, a bool in lower case."""
    def written(value):
        if isinstance(value, tuple):
            return "[" + ", ".join(written(v) for v in value) + "]"
        return value.kind if isinstance(value, Terms.Part) else str(value).lower() if isinstance(value, bool) else str(value)
    return ", ".join(f"{key}={written(value)}" for key, value in form.attributes.items())

<a id="T08.13.1"></a>
## 08.13.1 · A type keyword

```text
int        byte unsigned        real        string
```

The simplest data types are keywords: an integer atom (`byte`, `shortint`, `int`, `longint`, `integer`, `time`), signed
or unsigned; a non-integer type (`shortreal`, `real`, `realtime`); and the keyword types (`string`, `chandle`, `event`,
`void`). Each is a kind of `SystemVerilog Types`, `integeratomtype`, `nonintegertype` and `keywordtype`, its keyword and
its signing attributes, and each is the root of the dialect's holder, `S.SystemVerilog.Types.Expression`, which refines
the framework's `Expression` as every holder does (tutorial 3). The family's printer, `SystemVerilog.Text.ToText`, writes
any of its six dialects' terms (08.5 of tutorial 8):

In [2]:
with store.transaction():
    keywords = [S.SystemVerilog.Types.Expression(lambda e, root=root: e.root(root)) for root in [
        lambda u: u.integeratomtype(lambda a: a.keyword("int")),
        lambda u: u.integeratomtype(lambda a: a.keyword("byte").signing("unsigned")),
        lambda u: u.nonintegertype(lambda n: n.keyword("real")),
        lambda u: u.keywordtype(lambda k: k.keyword("string")),
    ]]
for held in keywords:
    form = Terms.form(Terms.term(held))
    print(f"{SystemVerilog.Text.ToText(held):14}", f"{form.kind:15}", f"{attributes(form):30}", Terms.dialect_of(Terms.term(held)))
assert [SystemVerilog.Text.ToText(held) for held in keywords] == ["int", "byte unsigned", "real", "string"]

int            integeratomtype keyword=int                    Expressions.SystemVerilog.Types.Term
byte unsigned  integeratomtype keyword=byte, signing=unsigned Expressions.SystemVerilog.Types.Term
real           nonintegertype  keyword=real                   Expressions.SystemVerilog.Types.Term
string         keywordtype     keyword=string                 Expressions.SystemVerilog.Types.Term


A keyword is one of a few the standard allows, and the dialect knows which: the choices are its kinds' own checks,
which validation reads (08.13.4).

<a id="T08.13.2"></a>
## 08.13.2 · Vectors and their dimensions

```text
logic signed [7:0]        bit [3:0]        logic [W-1:0]
```

A vector type, `bit`, `logic` or `reg`, is an `integervectortype`, its keyword and signing attributes again, and its
packed dimensions, each `[left:right]` a `rangedimension`. A dimension is no term: it is a helper, a value object the
type holds, which a form gives as a part (tutorial 6, 06.5). Its bounds are terms, and they are `Expressions`' terms,
expressions as any other: `W-1` is a binary expression whose left operand is a name. So a type can need a name from its
scope, as an expression does (tutorial 4):

In [3]:
with store.transaction():
    vectors = [S.SystemVerilog.Types.Expression(lambda e, root=root: e.root(root)) for root in [
        lambda u: u.integervectortype(lambda v: v.keyword("logic").signing("signed").dimensions([lambda d: d.rangedimension(lambda r: r.left(literal(7)).right(literal(0)))])),
        lambda u: u.integervectortype(lambda v: v.keyword("bit").dimensions([lambda d: d.rangedimension(lambda r: r.left(literal(3)).right(literal(0)))])),
        lambda u: u.integervectortype(lambda v: v.keyword("logic").dimensions([lambda d: d.rangedimension(lambda r: r
            .left(lambda b: b.binaryexpression(lambda o: o.left(name("W")).operator("-").right(literal(1)))).right(literal(0)))])),
    ]]
for held in vectors:
    form = Terms.form(Terms.term(held))
    print(f"{SystemVerilog.Text.ToText(held):19}", f"{attributes(form):48}", "needs",
          ", ".join(sorted(Symbolics.free(held, Types))) or "nothing")
bounds = Terms.form(Terms.term(vectors[2])).arguments
print("its bounds:", ", ".join(f"{SystemVerilog.Text.ToText(b)} ({Terms.kind_of(b)[0]} of {Terms.dialect_of(b)})" for b in bounds))
assert [SystemVerilog.Text.ToText(held) for held in vectors] == ["logic signed [7:0]", "bit [3:0]", "logic [W - 1:0]"]

logic signed [7:0]  keyword=logic, signing=signed, dimensions=[rangedimension] needs nothing
bit [3:0]           keyword=bit, dimensions=[rangedimension]         needs nothing
logic [W - 1:0]     keyword=logic, dimensions=[rangedimension]       needs W
its bounds: W - 1 (binaryexpression of Expressions.SystemVerilog.Expressions.Term), 0 (literal of Expressions.SystemVerilog.Expressions.Term)


The printer writes `W - 1` with spaces, as it writes any binary expression. A range is one of five kinds of dimension,
read from the store as the branches of the dialect's union `Expressions.SystemVerilog.Types.Dimension`: `[7:0]`, a
size `[8]`, an unsized `[]`, an associative `[int]` or `[*]` and a queue `[$]`:

In [4]:
print(", ".join(branch.name for branch in store.registered("Expressions.SystemVerilog.Types.Dimension").branches))

rangedimension, sizedimension, unsizeddimension, associativedimension, queuedimension


<a id="T08.13.3"></a>
## 08.13.3 · Structures, unions and enumerations: their members are data

```text
struct packed { logic [7:0] lo, hi; }
union tagged { void Invalid; int Valid; }
enum logic [1:0] {IDLE, RUN = 2}
word_t        type(x)
```

A structure or a union is a `structtype`, its keyword `struct` or `union`, `packed` or not, a tagged union's qualifier
`tagged`. Its members are data, not terms (stage 6, "Q7"): each a `structmember` helper holding its type, a term of this
dialect again, and its declarators, each a `variabledeclarator` naming a member. An enumeration is an `enumtype`, its
base type a term, its members `enummember` helpers, each a name and perhaps a value, an expression. A type named by a
typedef is a `namedtype`, its name `Expressions`' name helper (tutorial 4), and `type(x)`, a type reference, holds the
expression whose type it is:

In [5]:
byte_ = lambda t: t.integervectortype(lambda v: v.keyword("logic").dimensions([lambda d: d.rangedimension(lambda r: r.left(literal(7)).right(literal(0)))]))
declared = lambda spelling: lambda d: d.name(lambda i: i.spelling(spelling))  # a declarator, of its name
with store.transaction():
    aggregates = [S.SystemVerilog.Types.Expression(lambda e, root=root: e.root(root)) for root in [
        lambda u: u.structtype(lambda s: s.keyword("struct").packed(True).members([
            lambda m: m.type(byte_).declarators([declared("lo"), declared("hi")])])),
        lambda u: u.structtype(lambda s: s.keyword("union").qualifier("tagged").members([
            lambda m: m.type(lambda t: t.keywordtype(lambda k: k.keyword("void"))).declarators([declared("Invalid")]),
            lambda m: m.type(lambda t: t.integeratomtype(lambda a: a.keyword("int"))).declarators([declared("Valid")])])),
        lambda u: u.enumtype(lambda n: n
            .base(lambda t: t.integervectortype(lambda v: v.keyword("logic").dimensions([lambda d: d.rangedimension(lambda r: r.left(literal(1)).right(literal(0)))])))
            .members([lambda m: m.name(lambda i: i.spelling("IDLE")), lambda m: m.name(lambda i: i.spelling("RUN")).value(literal(2))])),
        lambda u: u.namedtype(lambda t: t.name(lambda n: n.identifier(lambda i: i.spelling("word_t")))),
        lambda u: u.typereference(lambda t: t.operand(name("x"))),
    ]]
for held in aggregates:
    form = Terms.form(Terms.term(held))
    print(f"{SystemVerilog.Text.ToText(held):42}", form.kind)
    print(" ", attributes(form) or "-", "| its terms:",
          ", ".join(Terms.kind_of(a)[0] for a in form.arguments if a is not None) or "none")
assert [SystemVerilog.Text.ToText(held) for held in aggregates] == [
    "struct packed { logic [7:0] lo, hi; }", "union tagged { void Invalid; int Valid; }",
    "enum logic [1:0] {IDLE, RUN = 2}", "word_t", "type(x)"]

struct packed { logic [7:0] lo, hi; }      structtype
  keyword=struct, packed=true, members=[structmember] | its terms: integervectortype
union tagged { void Invalid; int Valid; }  structtype
  keyword=union, qualifier=tagged, members=[structmember, structmember] | its terms: keywordtype, integeratomtype
enum logic [1:0] {IDLE, RUN = 2}           enumtype
  members=[enummember, enummember] | its terms: integervectortype, literal
word_t                                     namedtype
  name=identifier | its terms: none
type(x)                                    typereference
  - | its terms: nameexpression


A member's type is a subterm the structure's form gives among its arguments (`integervectortype`), while the member
itself, and its declarators' names, are in its attributes, as parts. A member's type may be any type, a structure
within a structure among them, so the dialect's `Term` holds them all.

<a id="T08.13.4"></a>
## 08.13.4 · Validated, and one refused

`validate` checks a type as it checks any term (tutorial 4): what each kind requires, each keyword among its choices,
each name's spelling, and every name its expressions read, which the caller must bind (the second column).
`logic [W-1:0]` needs `W`; given it, it is valid. A structure with a qualifier is refused, since only a union is
`tagged` or `soft` (IEEE 1800 7.3); the union above, the same members under `union`, passes:

In [6]:
with store.transaction():
    tagged_struct = S.SystemVerilog.Types.Expression(lambda e: e.root(lambda u: u.structtype(lambda s: s
        .keyword("struct").qualifier("tagged").members([
            lambda m: m.type(lambda t: t.integeratomtype(lambda a: a.keyword("int"))).declarators([declared("Valid")])]))))
for held, bound in [(vectors[2], ()), (vectors[2], {"W"}), (tagged_struct, ()), (aggregates[1], ())]:
    print(f"{SystemVerilog.Text.ToText(held):42}", f"{', '.join(bound) or '-':6}", "; ".join(Types.validate(held, bound)) or "valid")
assert Types.validate(tagged_struct) == ["a StructType with a qualifier is a union"]

logic [W - 1:0]                            -      dimensions 0 left: left: nameexpression 'W' is not bound
logic [W - 1:0]                            W      valid
struct tagged { int Valid; }               -      a StructType with a qualifier is a union
union tagged { void Invalid; int Valid; }  -      valid


Validation reports; it does not repair: the refused structure is kept, and printed as written, as tutorial 4 kept a name
its language could not spell.

<a id="T08.13.5"></a>
## 08.13.5 · Read by the expressions that hold it

```text
int'(x)    byte'(x)    real'(x)    logic [7:0]'(x)    $bits(logic [7:0])    pair_t'{1, 2}
```

A type stands where SystemVerilog's expressions put one: a cast's type, a system call's argument (`$bits` of a type), an
assignment pattern's type. Which properties of `SystemVerilog`'s kinds hold this dialect's `Term` is read from the store,
as tutorial 8 did (08.3). Its host, `SystemVerilog`'s `Expressions`, reads the type through the dialects it reads
(tutorial 8, 08.1), and validates the whole tree:

In [7]:
union = store.registered(Types.union)
host = store.registered(SystemVerilog.Expressions.DIALECT.union).branches
print(", ".join(f"{b.name}.{p}" for b in host for p, prop in b.schema.properties.items()
                if union in (prop.schema, getattr(prop.schema, "element", None))))
with store.transaction():
    held = [S.SystemVerilog.Expressions.Expression(lambda e, root=root: e.root(root)) for root in [
        lambda u: u.castexpression(lambda c: c.type(lambda t: t.integeratomtype(lambda a: a.keyword("int"))).value(name("x"))),
        lambda u: u.castexpression(lambda c: c.type(lambda t: t.integeratomtype(lambda a: a.keyword("byte"))).value(name("x"))),
        lambda u: u.castexpression(lambda c: c.type(lambda t: t.nonintegertype(lambda n: n.keyword("real"))).value(name("x"))),
        lambda u: u.castexpression(lambda c: c.type(byte_).value(name("x"))),
        lambda u: u.systemcall(lambda c: c.name("$bits").arguments([byte_])),
        lambda u: u.assignmentpattern(lambda a: a.type(lambda t: t.namedtype(lambda n: n.name(
            lambda m: m.identifier(lambda i: i.spelling("pair_t"))))).items([literal(1), literal(2)])),
    ]]
for expression in held:
    form = Terms.form(Terms.term(expression))
    types = [Terms.kind_of(a)[0] for a in form.arguments if a is not None and Terms.dialect_of(a) == Types.union]
    print(f"{SystemVerilog.Text.ToText(expression):19}", f"{form.kind:18}", "holds", ", ".join(types),
          "| problems:", ", ".join(SystemVerilog.Expressions.DIALECT.validate(expression, {"x"})) or "none")
assert [SystemVerilog.Text.ToText(e) for e in held] == [
    "int'(x)", "byte'(x)", "real'(x)", "logic [7:0]'(x)", "$bits(logic [7:0])", "pair_t'{1, 2}"]

assignmentpattern.type, patternitem.key, systemcall.arguments, castexpression.type, streamingconcatenation.slice
int'(x)             castexpression     holds integeratomtype | problems: none
byte'(x)            castexpression     holds integeratomtype | problems: none
real'(x)            castexpression     holds nonintegertype | problems: none
logic [7:0]'(x)     castexpression     holds integervectortype | problems: none
$bits(logic [7:0])  systemcall         holds integervectortype | problems: none
pair_t'{1, 2}       assignmentpattern  holds namedtype | problems: none


Its rules convert a value to a type they have a stored domain for, by name: SystemVerilog's own domains (tutorial 2,
02.4), `int` and `byte` here, so `int'(300)` is 300 and `byte'(300)` is 44, 300 wrapped to 8 bits. Its inference reads
the same domain, `real`'s too. A type it has no stored domain for, a vector with dimensions, a named type, stays in the residual, named by its kind and the dialect that owns it, as tutorial 8 showed every sublanguage's kinds are; its inference
gives `any`, no domain known, until each term's domain is inferred as data
([CAP-232: domains inferred for terms](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-232.md)):

In [8]:
dialect, evaluate = SystemVerilog.Expressions.DIALECT, SystemVerilog.Simplifiers.OfAny
int_ = Domains.domains(store.Instances.Expressions.SystemVerilog.Expressions)["int"]
for expression in held[:4]:
    print(f"{SystemVerilog.Text.ToText(expression):19}", f"{dialect.infer(expression, {'x': int_}).name():5}",
          refused(lambda e: evaluate(e, {"x": 300}), expression) if expression is not held[2] else "-")
print(f"{SystemVerilog.Text.ToText(held[4]):19}", f"{'':5}", refused(evaluate, held[4]))

int'(x)             int   32'sb00000000000000000000000100101100
byte'(x)            byte  8'sb00101100
real'(x)            real  -
logic [7:0]'(x)     any   SystemVerilog Types integervectortype is not evaluated
$bits(logic [7:0])        SystemVerilog Types integervectortype is not evaluated


(`real'(300)` folds too, to the float 300.0; it is left out here, since each language writes a float its own way.)

<a id="T08.13.6"></a>
## 08.13.6 · A type has no domain and no value of its own

A type names a set of values; it is not one. So a term of `SystemVerilog Types` has no domain to infer, `any`, and SystemVerilog's rules leaves it in the residual, named by its kind, held alone or wherever it stands. What it names is stored
data: the domain `int`, read by name, is the one the cast's inference gave above, 32 bits, signed, two-state:

In [9]:
print(Types.infer(keywords[0]).name(), "|", refused(evaluate, keywords[0]), "|", refused(evaluate, aggregates[0]))
properties = Domains.properties(int_)
print("int:", ".".join(Domains.kind_of(int_)[:2]), properties["width"], "bits,", "signed" if properties["signed"] else "unsigned")
assert Types.infer(keywords[0]) is Domains.Anything and dialect.infer(held[0], {"x": int_}).domain is int_

any | SystemVerilog Types integeratomtype is not evaluated | SystemVerilog Types structtype is not evaluated
int: systemverilog.bit 32 bits, signed


<a id="T08.13.7"></a>
## 08.13.7 · Its editions are data

The dialect covers Verilog's and SystemVerilog's editions, IEEE Std 1364-1995 to 1364-2005 and IEEE Std 1800-2005 to
1800-2023, each found by its designation or its common name, and each records the kinds it has
([CAP-268: standards as tagged data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-268.md)):
Verilog's have the keyword and vector types and the implicit type (`signed'(x)`), and IEEE 1800-2005 added named types,
virtual interfaces, structures, unions, enumerations and type references:

In [10]:
singleton = store.Instances.Expressions.SystemVerilog.Types
kinds = {branch.schema: branch.name for branch in store.registered(Types.union).branches}
for designation, edition in Standards.standards(singleton).items():
    print(f"{designation:19}", f"{Standards.tags(edition)[1]:19}", len(Standards.kinds(edition)), "kinds")
added = [kinds[k] for k in Standards.kinds(Standards.find(singleton, "SystemVerilog-2005"))
         if k not in Standards.kinds(Standards.find(singleton, "Verilog-2005"))]
print("added in IEEE 1800-2005:", ", ".join(added))
assert Standards.find(singleton, "IEEE Std 1800-2023") is Standards.find(singleton, "SystemVerilog-2023")

IEEE Std 1364-1995  Verilog-1995        5 kinds
IEEE Std 1364-2001  Verilog-2001        5 kinds
IEEE Std 1364-2005  Verilog-2005        5 kinds
IEEE Std 1800-2005  SystemVerilog-2005  10 kinds
IEEE Std 1800-2009  SystemVerilog-2009  10 kinds
IEEE Std 1800-2012  SystemVerilog-2012  10 kinds
IEEE Std 1800-2017  SystemVerilog-2017  10 kinds
IEEE Std 1800-2023  SystemVerilog-2023  10 kinds
added in IEEE 1800-2005: namedtype, virtualinterfacetype, structtype, enumtype, typereference


An edition records the kinds it has, not each keyword: `integeratomtype` is in Verilog-1995's, though `int` and `byte`
arrived with SystemVerilog.

## What you learned

- **A data type is a term of `SystemVerilog Types`**: a keyword, a vector and its dimensions, a structure, a union, an
  enumeration, a named type or a type reference, each held alone by the dialect's holder and printed by the family's
  printer.
- **Its parts are data where they are no type**: dimensions, members and declarators are helpers; a dimension's bounds
  and a member's value are `Expressions`' terms, so a type can need a name from its scope.
- **It is validated as any term**, its keywords' choices and its kinds' rules among its checks, and refused by name,
  never repaired.
- **Its host reads it**: a cast, `$bits` and an assignment pattern hold it; the rules convert to a type they have a stored domain for, and leaves the others in the residual, named.
- **A type has no domain or value of its own**, and the editions that have each of its kinds are data.

Next, the patterns a `matches` compares a value with: [SystemVerilog Patterns](08_14_SystemVerilog_Patterns.ipynb).

<!-- nav -->
[← 8.12 · POSIX ERE](08_12_POSIX_ERE.ipynb) · [Home](../../README.md) · [8.14 · SystemVerilog Patterns →](08_14_SystemVerilog_Patterns.ipynb)